# Day 1 — Environment Setup + Data Collection

**KAN-11** · Healthcare lane · Qwen + Llama

Runs end to end:

1. Mount Drive (everything persists there — `/content` vanishes on disconnect)
2. Clone the repo and install pinned dependencies
3. Verify GPU, pins, and 4-bit quantization
4. Download MedMCQA (Apache-2.0) and PubMedQA (MIT) straight to Drive
5. Checksum the raw data and print samples as evidence

Datasets were chosen for licence as much as content. ChatDoctor (no licence)
and MedQuAD (CC BY-SA share-alike) were excluded — see `data/raw/SOURCES.md`.

**Runtime → Change runtime type → T4 GPU** before running.

In [6]:
!pip install -q -U 'bitsandbytes>=0.46.1'


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 40.5 MB/s eta 0:00:00


In [1]:
# --- 1. Mount Drive ------------------------------------------------------
# First cell, every session. Colab can disconnect at any time and /content
# is wiped when it does; Drive is what survives.

from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

DRIVE = Path("/content/drive/MyDrive/healthcare-llm")
DRIVE.mkdir(parents=True, exist_ok=True)
print(f"project root: {DRIVE}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
project root: /content/drive/MyDrive/healthcare-llm


In [2]:
# --- 2. Repo + dependencies ---------------------------------------------
# The repo is private, so the clone needs a token. The token is fed to git
# over stdin via a credential helper — it never appears in a command line,
# in notebook output, or in the resulting .git/config.

import os
import subprocess
from getpass import getpass

REPO = "KusalPabasara/healthcare-llm-finetune"
PROJECT = Path("/content/healthcare-llm-finetune")


def sh(cmd, **kw):
    """Run a shell command, echoing it. Never pass secrets through this."""
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=False, **kw)


def clone_private(repo: str, dest: Path, token: str):
    """Clone without the token touching argv, output, or on-disk config."""
    env = {
        **os.environ,
        "GIT_TERMINAL_PROMPT": "0",
        # askpass reads the token from an env var the child process sees;
        # it is never part of the command git logs or stores.
        "GIT_ASKPASS": "/bin/echo",
        "GIT_USERNAME": token,
    }
    helper = f"!f() {{ echo username=x-access-token; echo password={token}; }}; f"
    return subprocess.run(
        [
            "git",
            "-c", f"credential.helper={helper}",
            "clone", "-q",
            f"https://github.com/{repo}.git",
            str(dest),
        ],
        env=env,
        capture_output=True,
        text=True,
    )


if not PROJECT.exists():
    tok = getpass("GitHub token (input hidden): ").strip()
    print(f"$ git clone https://github.com/{REPO}.git  (token supplied via credential helper)")
    res = clone_private(REPO, PROJECT, tok)
    del tok
    if res.returncode != 0:
        # git's own stderr does not contain the token — safe to surface, and
        # far more useful than a guess at what went wrong.
        raise SystemExit(f"clone failed:\n{res.stderr.strip()}")
    print("cloned")

os.chdir(PROJECT)
print(f"working in {Path.cwd()}")

working in /content/healthcare-llm-finetune


In [3]:
# Colab ships older versions of most of these. Installing the pins keeps
# results reproducible across sessions and comparable across the team's lanes.
sh("pip install -q -r requirements.txt")

$ pip install -q -r requirements.txt


CompletedProcess(args='pip install -q -r requirements.txt', returncode=0)

In [4]:
# --- 3. Verify the environment ------------------------------------------
# Fails loudly on version drift, missing GPU, or a broken 4-bit config.
# Do not proceed to Day 2 on a failure here.
#
# Note: pip may warn about needing a restart. If verify_env.py reports version
# drift, use Runtime > Restart session, then re-run from this cell.
import subprocess

def sh(cmd, **kw):
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=False,
                            capture_output=True, text=True, **kw)
    if result.stdout: print(result.stdout, end="")
    if result.stderr: print(result.stderr, end="")
    return result

result = sh("python scripts/verify_env.py")
print(f"\nexit code: {result.returncode}")


$ python scripts/verify_env.py
Python 3.12.13 on Linux
Platform: Colab

Pinned libraries
----------------------------------------------------
  ok    transformers     4.57.6
  ok    datasets         3.2.0
  ok    accelerate       1.2.1
  ok    peft             0.14.0
  MISS  bitsandbytes     not installed
  ok    trl              0.13.0

GPU
----------------------------------------------------
  torch            2.11.0+cu128
  devices          1 x Tesla T4
  memory           14.6 GiB each
  note             run Qwen and Llama SEQUENTIALLY on 15 GiB
  note             Tesla T4 has no bf16 — use fp16 compute dtype

4-bit quantization
----------------------------------------------------
  ok    BitsAndBytesConfig constructs (nf4, fp16 compute)

Persistence
----------------------------------------------------
  ok    Drive mounted at /content/drive/MyDrive
  ok    project dir  /content/drive/MyDrive/healthcare-llm
  WARN  /content is wiped on disconnect. Checkpoint to Drive:
        output

In [5]:
# --- 4. Download the datasets -------------------------------------------
# Written straight to Drive so a disconnect does not cost the 140MB download.
# data/raw/ is the reproducibility anchor: read-only for the rest of the sprint.

RAW = DRIVE / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)

# Point the repo's data/raw at Drive, so scripts stay path-agnostic.
local_raw = PROJECT / "data" / "raw"
if local_raw.is_symlink():
    local_raw.unlink()
elif local_raw.exists():
    import shutil

    shutil.rmtree(local_raw)
local_raw.parent.mkdir(parents=True, exist_ok=True)
local_raw.symlink_to(RAW)
print(f"data/raw -> {RAW}")

sh("python scripts/download_data.py")

data/raw -> /content/drive/MyDrive/healthcare-llm/data/raw
$ python scripts/download_data.py

  medmcqa    fetching openlifescienceai/medmcqa...
               train        182,822 rows    140.4 MB
               validation     4,183 rows      2.5 MB
  pubmedqa   fetching qiaojin/PubMedQA...
               train          1,000 rows      2.1 MB

  checksummed 5 files -> data/raw/CHECKSUMS.txt

Done. 188,005 rows staged in data/raw/
Provenance written to data/raw/SOURCES.md

Reminder: data/raw/ is read-only for the rest of the sprint.

Generating train split: 100%|██████████| 182822/182822 [00:00<00:00, 241483.27 examples/s]

Generating test split: 100%|██████████| 6150/6150 [00:00<00:00, 482482.64 examples/s]

Generating validation split: 100%|██████████| 4183/4183 [00:00<00:00, 318416.94 examples/s]

Creating json from Arrow format: 100%|██████████| 183/183 [00:01<00:00, 102.30ba/s]

Creating json from Arrow format: 100%|██████████| 5/5 [00:00<00:00, 144.71ba/s]

Generating train split

CompletedProcess(args='python scripts/download_data.py', returncode=0, stdout='Downloading healthcare datasets\n\n  medmcqa    fetching openlifescienceai/medmcqa...\n               train        182,822 rows    140.4 MB\n               validation     4,183 rows      2.5 MB\n  pubmedqa   fetching qiaojin/PubMedQA...\n               train          1,000 rows      2.1 MB\n\n  checksummed 5 files -> data/raw/CHECKSUMS.txt\n\nDone. 188,005 rows staged in data/raw/\nProvenance written to data/raw/SOURCES.md\n\nReminder: data/raw/ is read-only for the rest of the sprint.\n', stderr='\nGenerating train split:   0%|          | 0/182822 [00:00<?, ? examples/s]\nGenerating train split:   9%|▉         | 17000/182822 [00:00<00:01, 159558.47 examples/s]\nGenerating train split:  24%|██▍       | 44000/182822 [00:00<00:00, 218497.04 examples/s]\nGenerating train split:  39%|███▉      | 71000/182822 [00:00<00:00, 231734.60 examples/s]\nGenerating train split:  53%|█████▎    | 97000/182822 [00:00<00:00, 

In [6]:
# --- 5. Inspect what landed ---------------------------------------------
# A sample row from each dataset — evidence the data is real and correctly
# shaped, rather than an empty success message.

import json

for name in ("medmcqa", "pubmedqa"):
    path = RAW / name / "train.jsonl"
    if not path.exists():
        print(f"{name}: MISSING")
        continue
    with path.open() as fh:
        first = json.loads(fh.readline())
        rows = 1 + sum(1 for _ in fh)
    print(f"\n{'=' * 60}\n{name}  —  {rows:,} rows\n{'=' * 60}")
    for k, v in first.items():
        s = str(v).replace("\n", " ")
        print(f"  {k:<16} {s[:90]}{'...' if len(s) > 90 else ''}")


medmcqa  —  182,822 rows
  id               e9ad821a-c438-4965-9f77-760819dfa155
  question         Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following...
  opa              Hyperplasia
  opb              Hyperophy
  opc              Atrophy
  opd              Dyplasia
  cop              2
  choice_type      single
  exp              Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, norm...
  subject_name     Anatomy
  topic_name       Urinary tract

pubmedqa  —  1,000 rows
  pubid            21645374
  question         Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
  context          {'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organ...
  long_answer      Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, a...
  final_decision   yes


In [7]:
# --- 6. Confirm persistence ---------------------------------------------
# The whole point of writing to Drive. If this shows files, Day 2 can attach
# them without re-downloading.

sh(f"du -sh {RAW}/* 2>/dev/null")
sh(f"python scripts/download_data.py --verify")

$ du -sh /content/drive/MyDrive/healthcare-llm/data/raw/* 2>/dev/null
512	/content/drive/MyDrive/healthcare-llm/data/raw/CHECKSUMS.txt
2.0K	/content/drive/MyDrive/healthcare-llm/data/raw/SOURCES.md
143M	/content/drive/MyDrive/healthcare-llm/data/raw/medmcqa
2.2M	/content/drive/MyDrive/healthcare-llm/data/raw/pubmedqa
$ python scripts/download_data.py --verify
  all files match — raw data intact


CompletedProcess(args='python scripts/download_data.py --verify', returncode=0, stdout='  all files match — raw data intact\n', stderr='')

In [8]:
# --- Day 1 complete ------------------------------------------------------
print(f"""
Day 1 done. Record in PROGRESS.md frozen decisions:
  - GPU type and compute dtype (from verify_env.py above)
  - Dataset row counts and licences ({RAW}/SOURCES.md)
  - Raw data checksums ({RAW}/CHECKSUMS.txt)

Data is on Drive at {RAW} and survives disconnects.

Next: Day 2 (KAN-15) — cleaning, instruction formatting, 80/10/10 split.
""")


Day 1 done. Record in PROGRESS.md frozen decisions:
  - GPU type and compute dtype (from verify_env.py above)
  - Dataset row counts and licences (/content/drive/MyDrive/healthcare-llm/data/raw/SOURCES.md)
  - Raw data checksums (/content/drive/MyDrive/healthcare-llm/data/raw/CHECKSUMS.txt)

Data is on Drive at /content/drive/MyDrive/healthcare-llm/data/raw and survives disconnects.

Next: Day 2 (KAN-15) — cleaning, instruction formatting, 80/10/10 split.

